## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [3]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        ####################
        #第一层: 输入维度784，输出维度100
        self.W1 = tf.Variable(tf.random.normal([784, 100]), name='weights1')
        self.b1 = tf.Variable(tf.zeros([100]), name='bias1')
        #第二层: 输入维度100，输出维度10
        self.W2 = tf.Variable(tf.random.normal([100, 10]), name='weights2')
        self.b2 = tf.Variable(tf.zeros([10]), name='bias2')
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        ####################
        #将输入展平 (batch_size, 28, 28) -> (batch_size, 784)
        x = tf.reshape(x, (-1, 784))
        #线性变换 + ReLU
        h1 = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)
        #线性变换，得到logits
        logits = tf.matmul(h1, self.W2) + self.b2
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [4]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [5]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 85.1197 ; accuracy 0.09571667
epoch 1 : loss 75.87823 ; accuracy 0.101783335
epoch 2 : loss 70.20775 ; accuracy 0.10695
epoch 3 : loss 66.42258 ; accuracy 0.113466665
epoch 4 : loss 63.6635 ; accuracy 0.11838333
epoch 5 : loss 61.488876 ; accuracy 0.123516664
epoch 6 : loss 59.67095 ; accuracy 0.12861666
epoch 7 : loss 58.087234 ; accuracy 0.13308333
epoch 8 : loss 56.662743 ; accuracy 0.13796666
epoch 9 : loss 55.357235 ; accuracy 0.14311667
epoch 10 : loss 54.141617 ; accuracy 0.14731666
epoch 11 : loss 52.996418 ; accuracy 0.15171666
epoch 12 : loss 51.9092 ; accuracy 0.15616667
epoch 13 : loss 50.870743 ; accuracy 0.16098334
epoch 14 : loss 49.874855 ; accuracy 0.1651
epoch 15 : loss 48.917187 ; accuracy 0.16968334
epoch 16 : loss 47.993717 ; accuracy 0.17391667
epoch 17 : loss 47.101967 ; accuracy 0.17775
epoch 18 : loss 46.239887 ; accuracy 0.18166667
epoch 19 : loss 45.406517 ; accuracy 0.18546666
epoch 20 : loss 44.60256 ; accuracy 0.18976666
epoch 21 : loss 43.8